# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Raja-saab/Flyrank1/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

## 1. My lane as an ML task (type)

My chosen lane is Refresh / Content Opportunity Scoring.

I would frame this primarily as a ranking task. The goal is not simply to classify every page as "good" or "bad." The practical decision is to determine which pages should be reviewed first when the available content/SEO review capacity is limited.

A ranking approach fits the lane because the final output is a prioritized queue of pages. Pages with stronger evidence of decline or opportunity should appear higher in the queue, while lower-priority pages should appear later.

The starter pipeline uses a decline label as a proxy and produces a ranked review queue. The lane guide also recommends ranking metrics such as Precision@K, recall, and average precision because they match the way the output is actually used.

The human action supported by the ranking is to review a page and decide whether it should be refreshed, expanded, protected, pruned, or monitored. The ranking is therefore a **decision-support tool**, not an automatic instruction to change a page.


In [5]:
import pandas as pd

DATA_PATH = "/content/Flyrank1/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nTrend direction:")
print(df["trend_direction"].value_counts())

Rows: 30000
Columns: 44

Trend direction:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

For the starter version of this task, I would use:

`is_declining_label = trend_direction == "down"`

as the target/proxy.

This label represents whether a content page is currently classified as having a declining trend. It comes from an observed field in the starter dataset rather than from a FlyRank product decision.

I consider this a **proxy label**, not the ideal final target. It is calculated from the current data window, so it does not answer the stronger question of whether a page will decline in a future period.

A stronger future version of the task would use a time-separated target such as:

**features from a prior 90-day window → decline or recovery during the next 30 days**

That future-looking setup would better match the real decision because the model would use information available before the outcome it is trying to predict.

For this Week 2 framing exercise, however, I will use the starter `trend_direction == "down"` label so that the task can be demonstrated on the provided dataset while clearly acknowledging its limitation.


In [6]:
# Sketch the starter proxy target

df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print("Target counts:")
print(df["is_declining_label"].value_counts())

print("\nTarget percentages:")
print(
    df["is_declining_label"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Target counts:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Target percentages:
is_declining_label
1    54.21
0    45.79
Name: proportion, dtype: float64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

My primary success metric will be **Precision@50**.

Precision@50 measures how many of the top 50 pages selected by the ranking are positive according to the target.

This metric matches the real action because the output is intended to be a limited review queue. If a team can review 50 pages, a useful ranking should place as many genuinely relevant pages as possible near the top.

For example, a Precision@50 of 0.70 would mean that approximately 35 of the top 50 recommended pages were positive according to the chosen label.

I prefer Precision@50 over generic accuracy because the task is not to classify every page equally. The important question is whether the **highest-priority pages are useful to review first**.

I would compare the ML ranking against a transparent fixed-rule baseline. A model should only be considered useful if its improvement is meaningful for the actual review decision.


In [7]:
# Starter benchmark from the repository documentation

baseline_precision_at_50 = 0.240
random_forest_precision_at_50 = 0.740

print(f"Baseline Precision@50: {baseline_precision_at_50:.3f}")
print(f"Random forest Precision@50: {random_forest_precision_at_50:.3f}")

print(
    f"\nRandom forest improvement: "
    f"{random_forest_precision_at_50 - baseline_precision_at_50:.3f}"
)

Baseline Precision@50: 0.240
Random forest Precision@50: 0.740

Random forest improvement: 0.500


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

The unit of analysis is **one content page**.

Each row represents one pseudonymized content item and contains observable signals about its search performance, content characteristics, engagement, and trend.

For this lane, the model would assign a score or ranking to each page. The resulting ranking would then determine which pages should receive human review first.

The important identifiers are `content_id` and `client_id`. These are pseudonymized identifiers and are used to distinguish records rather than to represent meaningful names.

The starter model uses observable signals such as search volume, impressions, clicks, sessions, content age, CTR, average position, engagement, scroll rate, and content characteristics. These are potential inputs because they describe observed data rather than FlyRank's own product decision.

The target/proxy is `is_declining_label`, which I define from `trend_direction == "down"`.


In [8]:
# Show the unit of analysis as a real dataframe

page_slice = df[
    [
        "content_id",
        "client_id",
        "search_volume",
        "impressions_90d",
        "sessions_90d",
        "content_age_days",
        "ctr",
        "avg_position",
        "trend_direction",
        "is_declining_label",
    ]
].copy()

print("One row = one content page")
print("Rows:", len(page_slice))

display(page_slice.head(10))

One row = one content page
Rows: 30000


,content_id,client_id,search_volume,impressions_90d,sessions_90d,content_age_days,ctr,avg_position,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,10.0,3803,17,187,0.76,10.6,down,1
1,content_a1fb4e703a9e,client_4e07408562,90.0,15320,9,445,0.05,20.3,down,1
2,content_9aa793d4d895,client_7f2253d7e2,0.0,12581,11,141,0.09,36.5,down,1
3,content_331d6c4de07b,client_19581e27de,10.0,11751,78,463,0.49,6.2,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,19140,145,263,0.13,44.0,down,1
5,content_d4084a4bc775,client_f369cb89fc,720.0,3970,5,147,0.03,8.5,down,1
6,content_9a34b442b552,client_8722616204,0.0,20,1,90,0.00,7.0,down,1
7,content_a63219c6e95a,client_19581e27de,590.0,1724,28,445,0.06,21.2,stable,0
8,content_5e6c160719bc,client_6208ef0f77,0.0,32574,68,90,0.09,46.0,down,1
9,content_c27558df2b0c,client_19581e27de,0.0,1240,3,257,0.16,4.9,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule is useful as a baseline because it is simple and easy to explain. However, the refresh opportunity problem involves many signals that can interact in different ways.

For example, a page with a declining trend may deserve more attention if it also has meaningful search demand, strong visibility, an older content age, or a weak CTR. Another declining page with almost no search demand may be much less important to review.

Writing every possible combination as an `if` statement would require many manually chosen thresholds and weights. Those thresholds may also work differently across different types of pages and clients.

ML can learn patterns across several observable signals and assign a probability or score that can be used to rank pages. This gives us a way to compare a learned approach against a transparent fixed-rule baseline.

The starter results provide evidence for this framing: the baseline rules achieved a Precision@50 of 0.240, while the random forest achieved 0.740 on the starter slice. This suggests that a learned ranking can improve prioritization over the fixed rule in this dataset.

However, this does not prove that ML will always outperform rules, nor does it prove that the recommended page will recover after a refresh. The model should remain a decision-support tool, and its results need to be validated honestly.


In [9]:
# Compare the starter fixed-rule baseline with the starter ML result

results = pd.DataFrame({
    "method": ["Baseline rules", "Random forest"],
    "precision_at_50": [0.240, 0.740]
})

display(results)

lift = (
    results.loc[1, "precision_at_50"]
    / results.loc[0, "precision_at_50"]
)

print(f"Relative Precision@50 improvement: {lift:.2f}x")

,method,precision_at_50
0,Baseline rules,0.24
1,Random forest,0.74


Relative Precision@50 improvement: 3.08x


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.